# 05a · JEV Tut

03에서 확정한 compact feature 20개를 같은 작은 validation sample에 입력해 RAW, train-only Z-score, train-only Robust 표현의 JEV 1.13 반응을 살펴봅니다. 성능 증명이나 Huang 2018 benchmark가 아닙니다.

03의 fixed grouped split과 selected features를 재사용합니다. 5,000/50,000초 TTF 구간은 sample strata와 사후 평가에만 사용하고 API 요청에는 feature 이름/값만 넣습니다.

N_PER_REGION=5로 최대 3 targets × 3 regions × 5 samples × 3 representations = 135 inference calls이며, RAW 반복 6회까지 최대 141회입니다. Smoke 요청은 같은 cache key를 사용해 중복 호출하지 않습니다. Smoke 결과를 먼저 확인한 뒤 전체 실험을 켭니다.

OpenRouter Decisions API의 `typesafe/jev-1.13`을 사용합니다. JEV의 0–2 score를 2로 나누어 0–1 risk_score로 표시하고, confidence와 각 상태 확률은 JEV 원본 응답을 사용합니다.

이번 단계에서는 threshold 조정, retrieval, LLM Judge, human review, 최종 FabJudge 성능 주장을 하지 않습니다.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, math, os, time, urllib.request
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import spearmanr
from sklearn.metrics import average_precision_score, roc_auc_score

def find_root(start=None):
    for p in (Path(start or Path.cwd()).resolve(), *Path(start or Path.cwd()).resolve().parents):
        if (p / "artifacts/random_forest/jev_candidate_features.csv").is_file():
            return p
    raise FileNotFoundError("03 compact-feature artifact not found")

ROOT = find_root()
RF_DIR = ROOT / "artifacts/random_forest"
TRAIN_DIR = ROOT / "data/interim/phm2018_jev/features/train"
OUT = ROOT / "artifacts/jev_tut"
CACHE = OUT / "cache"
OUT.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)
SEED = 42
N_PER_REGION = 5
REGIONS = ("normal", "boundary", "near_failure")
REPRESENTATIONS = ("raw", "zscore", "robust")
MAX_API_CALLS = 150
RUN_SINGLE_SMOKE = True
RUN_REPRESENTATION_SMOKE = False  # Review the one-call JEV response before enabling six-call smoke.
RUN_FULL_EXPERIMENT = False  # Review smoke output, then set True.
PROMPT_VERSION = "jev_tut_v4"
MODEL_ID = "typesafe/jev-1.13"  # Pinned Jev model in OpenRouter's Decisions API.
DECISIONS_URL = "https://openrouter.ai/api/alpha/decisions"
DECISIONS_DOCS_URL = "https://openrouter.ai/docs/api/api-reference/alphadecisions/submit-a-decisions-request"
CHUNK_ROWS = 100_000
QUANTILE_STRIDE = 20
LABELS = {
    "ttf_flowcool_pressure_low_seconds":"FlowCool pressure low",
    "ttf_flowcool_pressure_high_seconds":"FlowCool pressure high",
    "ttf_flowcool_leak_seconds":"FlowCool leak",
}

def load_env_keys(path):
    if path.is_file():
        for line in path.read_text(encoding="utf-8").splitlines():
            s = line.strip()
            if s and not s.startswith("#") and "=" in s:
                k, v = s.split("=", 1)
                k, v = k.strip(), v.strip().strip('"').strip("'")
                if k in {"JEV_API_KEY", "OPENROUTER_API_KEY"} and v and not os.environ.get(k):
                    os.environ[k] = v

load_env_keys(ROOT / ".env")
API_KEY = os.environ.get("JEV_API_KEY") or os.environ.get("OPENROUTER_API_KEY")
print("Project:", ROOT)
print("OpenRouter credential available:", bool(API_KEY))
print("Pinned JEV 1.13 Decisions model ID:", MODEL_ID)
print("Expected core calls: 135; with repeatability: at most 141")

## 1. Existing artifact audit

Re-use the exact 03 selected compact features and group split. This notebook does not run feature selection again. Evaluation candidates are restricted to validation groups.

In [ ]:
split_path = RF_DIR / "ttf_baseline_group_split.csv"
meta_path = RF_DIR / "ttf_baseline_split_metadata.json"
feature_path = RF_DIR / "jev_candidate_features.csv"
if not all(p.is_file() for p in (split_path, meta_path, feature_path)):
    raise FileNotFoundError("Run the saved artifact cells in notebooks/03_rf_feature_check.ipynb first.")
meta = json.loads(meta_path.read_text(encoding="utf-8"))
groups = pd.read_csv(split_path, low_memory=False)
features = pd.read_csv(feature_path)["feature"].astype(str).tolist()
targets = list(meta["target_columns"])
if len(features) != 20 or len(set(features)) != 20:
    raise ValueError(f"Expected 20 unique 03 compact features; got {len(features)}")
if targets != list(LABELS) or int(meta["random_seed"]) != SEED or int(meta["group_overlap"]) != 0:
    raise ValueError("Saved 03 target or fixed split metadata did not pass audit.")
split_lookup = dict(zip(groups.group_id.astype(str), groups.split.astype(str)))
train_ids = {g for g,s in split_lookup.items() if s == "train"}
valid_ids = {g for g,s in split_lookup.items() if s == "validation"}
assert not (train_ids & valid_ids)
paths = sorted(TRAIN_DIR.glob("*.csv"))
if not paths:
    raise FileNotFoundError(f"No processed training feature CSVs under {TRAIN_DIR}")
header = pd.read_csv(paths[0], nrows=0).columns.tolist()
usecols = ["source_file","Tool","sequence_index","window_index",*features,*targets]
if set(usecols) - set(header):
    raise ValueError(f"CSV lacks columns: {sorted(set(usecols)-set(header))}")
for p in paths[1:]:
    if pd.read_csv(p, nrows=0).columns.tolist() != header:
        raise ValueError(f"CSV header mismatch: {p.name}")
print("Targets:", targets, "| compact features:", len(features))
print("Fixed groups: train",len(train_ids),"validation",len(valid_ids),"overlap",len(train_ids&valid_ids))
print("Feature files:",len(paths))
display(pd.DataFrame({"selected compact feature":features}))

## 2. Train-only scaling statistics and representative sample

Stream only identifiers, the 20 compact features and TTF columns. Exact moments and extrema use 03 training groups. Quantiles use a deterministic one-in-twenty window-index subset of training rows. Validation rows are stratified by the measured TTF bands and balanced across sequence groups; no global random-row sampling is used.

In [ ]:
def region_of(a):
    a = np.asarray(a, dtype=float)
    out = np.full(a.shape, "", dtype=object)
    ok = np.isfinite(a)
    out[ok & (a > 50000)] = "normal"
    out[ok & (a > 5000) & (a <= 50000)] = "boundary"
    out[ok & (a <= 5000)] = "near_failure"
    return out

def balanced_quotas(counts, budget):
    counts = {str(k):int(v) for k,v in counts.items() if int(v)>0}
    quotas = {k:0 for k in counts}
    active = sorted(counts)
    remaining = min(int(budget),sum(counts.values()))
    while active and remaining:
        base, extra = divmod(remaining,len(active))
        if base == 0:
            for g in active[:remaining]: quotas[g] += 1
            break
        full = [g for g in active if counts[g]-quotas[g] <= base]
        if full:
            for g in full:
                n = counts[g]-quotas[g]
                quotas[g] += n
                remaining -= n
            full = set(full)
            active = [g for g in active if g not in full]
        else:
            for i,g in enumerate(active): quotas[g] += base + int(i < extra)
            remaining = 0
    return {g:n for g,n in quotas.items() if n}

acc = {t:{"n":np.zeros(len(features),dtype=np.int64),"sum":np.zeros(len(features)),
          "sq":np.zeros(len(features)),"min":np.full(len(features),np.inf),
          "max":np.full(len(features),-np.inf),"q":[[] for _ in features]} for t in targets}
strata_counts = {t:{r:{} for r in REGIONS} for t in targets}
total_rows = 0
for fi,path in enumerate(paths,1):
    print(f"Scan {fi}/{len(paths)}: {path.name}",flush=True)
    for d in pd.read_csv(path,usecols=usecols,chunksize=CHUNK_ROWS,low_memory=False):
        total_rows += len(d)
        gid = d.source_file.astype(str) + "::" + d.sequence_index.astype(str)
        split = gid.map(split_lookup)
        if split.isna().any(): raise ValueError("Unseen group in saved fixed split")
        is_train, is_valid = split.eq("train").to_numpy(), split.eq("validation").to_numpy()
        x = d[features].apply(pd.to_numeric,errors="coerce").replace([np.inf,-np.inf],np.nan).to_numpy(dtype=float)
        complete = np.isfinite(x).all(axis=1)
        wi = pd.to_numeric(d.window_index,errors="coerce").fillna(-1).to_numpy(dtype=np.int64)
        gid_values = gid.to_numpy()
        for t in targets:
            y = pd.to_numeric(d[t],errors="coerce").to_numpy(dtype=float)
            finite_y = np.isfinite(y)
            tr = is_train & finite_y
            if tr.any():
                v = x[tr]; ok = np.isfinite(v); a = acc[t]
                a["n"] += ok.sum(0)
                a["sum"] += np.where(ok,v,0).sum(0)
                a["sq"] += np.where(ok,v*v,0).sum(0)
                a["min"] = np.minimum(a["min"],np.where(ok,v,np.inf).min(0))
                a["max"] = np.maximum(a["max"],np.where(ok,v,-np.inf).max(0))
                qmask = tr & (wi >= 0) & (wi % QUANTILE_STRIDE == 0)
                if qmask.any():
                    qx = x[qmask]
                    for j in range(len(features)):
                        z = qx[:,j]; z = z[np.isfinite(z)]
                        if len(z): a["q"][j].append(z.copy())
            va = is_valid & finite_y & complete
            if va.any():
                vg, vr = gid_values[va], region_of(y[va])
                for r in REGIONS:
                    vals = pd.Series(vg[vr == r]).value_counts(sort=False)
                    for g,n in vals.items():
                        strata_counts[t][r][g] = strata_counts[t][r].get(g,0)+int(n)

scaler = {}
dist_rows = []
for t in targets:
    scaler[t] = {}
    for j,f in enumerate(features):
        a=acc[t]; n=int(a["n"][j])
        if not n: raise ValueError(f"No train values: {t}/{f}")
        mean=float(a["sum"][j]/n); std=math.sqrt(max(float(a["sq"][j]/n-mean*mean),0.0))
        q=np.concatenate(a["q"][j]) if a["q"][j] else np.array([])
        if not len(q): raise ValueError(f"No quantile sample: {t}/{f}")
        p01,q25,median,q75,p99=np.quantile(q,[.01,.25,.5,.75,.99])
        iqr=float(q75-q25)
        scaler[t][f]={"n_train_values":n,"quantile_sample_n":len(q),"min":float(a["min"][j]),
            "p01_approx":float(p01),"mean":mean,"std":std,"median_approx":float(median),
            "p99_approx":float(p99),"max":float(a["max"][j]),"q25_approx":float(q25),
            "q75_approx":float(q75),"iqr_approx":iqr,"safe_std":std if std>1e-12 else 1.0,
            "safe_iqr":iqr if iqr>1e-12 else 1.0}
        dist_rows.append({"target":t,"feature":f,**scaler[t][f]})
dist = pd.DataFrame(dist_rows)
(OUT/"train_scaler_stats.json").write_text(json.dumps({"source":"03 train groups only",
 "exact_moments_and_extrema":True,"quantiles":"deterministic window_index modulo 20 sample",
 "targets":scaler},indent=2,allow_nan=False),encoding="utf-8")
display(dist[["target","feature","n_train_values","min","p01_approx","median_approx","p99_approx","max","mean","std","iqr_approx"]].head(12))
print("Chunked rows scanned:",f"{total_rows:,}")

plan={}
for t in targets:
    for r in REGIONS:
        q=balanced_quotas(strata_counts[t][r],N_PER_REGION)
        plan[t,r]={g:np.linspace(0,strata_counts[t][r][g]-1,num=n,dtype=np.int64) for g,n in q.items()}

cursors={}
rows=[]
for fi,path in enumerate(paths,1):
    print(f"Select {fi}/{len(paths)}: {path.name}",flush=True)
    for d in pd.read_csv(path,usecols=usecols,chunksize=CHUNK_ROWS,low_memory=False):
        gid=d.source_file.astype(str)+"::"+d.sequence_index.astype(str)
        is_valid=gid.map(split_lookup).eq("validation").to_numpy()
        x=d[features].apply(pd.to_numeric,errors="coerce").replace([np.inf,-np.inf],np.nan).to_numpy(dtype=float)
        complete=np.isfinite(x).all(1); gidv=gid.to_numpy()
        for t in targets:
            y=pd.to_numeric(d[t],errors="coerce").to_numpy(dtype=float)
            valid=is_valid & np.isfinite(y) & complete
            if not valid.any(): continue
            reg=region_of(y)
            for r in REGIONS:
                pos=np.flatnonzero(valid & (reg==r))
                if not len(pos): continue
                frame=pd.DataFrame({"pos":pos,"gid":gidv[pos]})
                for g,sub in frame.groupby("gid",sort=False):
                    wanted=plan[t,r].get(g)
                    if wanted is None: continue
                    key=(t,r,g); start=cursors.get(key,0)
                    ords=start+np.arange(len(sub),dtype=np.int64)
                    keep=np.isin(ords,wanted,assume_unique=True)
                    if keep.any():
                        for p in sub.loc[keep,"pos"].to_numpy(dtype=int):
                            z=d.iloc[p]; source=str(z.source_file); seq=str(z.sequence_index); wi=int(z.window_index)
                            slug=t.removeprefix("ttf_").removesuffix("_seconds")
                            rows.append({"target":t,"target_label":LABELS[t],
                                "sample_id":f"{slug}|{source}|{seq}|w{wi}","source_file":source,
                                "Tool":str(z.Tool),"sequence_index":seq,"window_index":wi,
                                "ttf_seconds":float(y[p]),"region":r,
                                **{f:float(z[f]) for f in features}})
                    cursors[key]=start+len(sub)
samples=pd.DataFrame(rows)
if samples.empty or samples.sample_id.duplicated().any(): raise ValueError("Sample selection empty or IDs duplicated")
samples.to_csv(OUT/"selected_samples.csv",index=False)
sample_counts=samples.groupby(["target","region"]).size().unstack(fill_value=0).reindex(columns=REGIONS,fill_value=0)
print("Selected rows:",len(samples),"| requested per target and region:",N_PER_REGION)
display(sample_counts)

## 3. Ground-truth isolation and feature magnitude

TTF, region, target, sequence and tool identifiers remain in the local evaluation frame. The request builder below serializes only the 20 selected feature names and values.

In [ ]:
assert set(features).issubset(samples.columns)
assert samples.region.isin(REGIONS).all()
print("Only selected feature columns will be sent; TTF/region/target/IDs stay local.")
display(dist.groupby("feature",as_index=False).agg(
    train_min=("min","min"),train_p01=("p01_approx","median"),train_median=("median_approx","median"),
    train_p99=("p99_approx","median"),train_max=("max","max"),train_mean=("mean","median"),
    train_std=("std","median"),train_iqr=("iqr_approx","median")
).sort_values("train_max",ascending=False).head(12))

## 4. OpenRouter API/config audit

The credential is loaded from JEV_API_KEY (or OPENROUTER_API_KEY) in the environment or local .env. The key is never printed or saved. OpenRouter's [Decisions API reference](https://openrouter.ai/docs/api/api-reference/alphadecisions/submit-a-decisions-request) specifies the pinned `typesafe/jev-1.13` model, `POST /api/alpha/decisions`, and `model` / `state` / `questions` request fields. The single-call response confirms the returned model and TypeSafe provider.

In [ ]:
def openrouter_json(url,payload,timeout=45):
    if not API_KEY: raise RuntimeError("No OpenRouter API key is configured")
    headers={"Authorization":f"Bearer {API_KEY}","Content-Type":"application/json"}
    data=json.dumps(payload,ensure_ascii=False,allow_nan=False).encode("utf-8")
    req=urllib.request.Request(url,data=data,headers=headers,method="POST")
    with urllib.request.urlopen(req,timeout=timeout) as resp:
        return json.loads(resp.read().decode("utf-8"))

API_STATUS="available" if API_KEY else "unavailable"
API_ERROR=None if API_KEY else "No configured OpenRouter API key"
print("Credential configured:",bool(API_KEY),"| JEV Decisions endpoint:",DECISIONS_URL)
if API_ERROR: print("JEV API unavailable:",API_ERROR)

## 5. Prompt, input representations, cache and parser

Each request asks one native JEV `score` question with three ordered states. `risk_score = score / 2`; `state` is the most probable level; `confidence` comes from JEV. RAW keeps source float precision. Z-score and Robust use statistics fitted on 03 training groups only. The same sample IDs are used for all three variants.

In [ ]:
DECISION_QUESTION={"type":"score",
 "instructions":"Using only the numeric semiconductor equipment measurements in state.features, how abnormal or failure-like is this equipment state? Judge overall degradation, not time to failure.",
 "criteria":["Normal equipment state: measurements do not suggest a fault",
             "Uncertain equipment state: measurements may suggest degradation",
             "Strongly abnormal or failure-like equipment state: measurements suggest a fault"]}
(OUT/"prompt_v4.txt").write_text(
    "PROMPT_VERSION="+PROMPT_VERSION+"\nmodel="+MODEL_ID+"\nendpoint="+DECISIONS_URL+
    "\nquestion=\n"+json.dumps(DECISION_QUESTION,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")

def make_payload(sample,representation):
    t=str(sample.target); values={}
    for f in features:
        v=float(sample[f])
        if representation=="zscore":
            s=scaler[t][f]; v=(v-s["mean"])/s["safe_std"]
        elif representation=="robust":
            s=scaler[t][f]; v=(v-s["median_approx"])/s["safe_iqr"]
        elif representation!="raw": raise ValueError(representation)
        if not math.isfinite(v): raise ValueError(f"Non-finite input: {f}")
        values[f]=v
    payload={"features":values}
    assert len(values)==20 and set(payload)=={"features"}
    return payload

def parse_decision(raw):
    answer=(raw.get("answers") or {}).get("equipment_state") or {}
    try:
        if not str(raw.get("model","")).startswith(MODEL_ID): raise ValueError("returned model is not Jev 1.13")
        if raw.get("provider")!="TypeSafe": raise ValueError("returned provider is not TypeSafe")
        if answer.get("type")!="score": raise ValueError("unexpected decision type")
        score=float(answer["score"]); conf=float(answer["confidence"])
        probs={str(i):float(answer["probabilities"][str(i)]) for i in range(3)}
        if not (math.isfinite(score) and 0<=score<=2): raise ValueError("score outside [0,2]")
        if not (math.isfinite(conf) and 0<=conf<=1): raise ValueError("confidence outside [0,1]")
        if any(not math.isfinite(v) or not 0<=v<=1 for v in probs.values()): raise ValueError("invalid probabilities")
        if not .95<=sum(probs.values())<=1.05: raise ValueError("probabilities do not sum to one")
    except (KeyError,TypeError,ValueError) as e: return answer,False,str(e)
    state=("normal","uncertain","degrading")[max(range(3),key=lambda i:probs[str(i)])]
    return {"risk_score":score/2,"confidence":conf,"state":state,
            "native_score":score,"probabilities":probs},True,None

def cache_file(sample,rep,repeat_tag=""):
    key="|".join([str(sample.target),str(sample.sample_id),rep,PROMPT_VERSION,MODEL_ID,repeat_tag])
    return CACHE/(hashlib.sha256(key.encode()).hexdigest()+".json")

CALLS=0
CACHE_HITS=0
API_ERRORS=0
def call_jev(sample,rep,repeat_tag=""):
    global CALLS,CACHE_HITS,API_ERRORS
    path=cache_file(sample,rep,repeat_tag)
    if path.is_file():
        saved=json.loads(path.read_text(encoding="utf-8")); saved["cache_hit"]=True; CACHE_HITS+=1
        return saved
    stamp=datetime.now(timezone.utc).isoformat()
    if API_STATUS!="available" or CALLS>=MAX_API_CALLS:
        return {"sample_id":str(sample.sample_id),"representation":rep,"request_timestamp":stamp,
                "raw_response":None,"raw_content":None,"parsed_response":None,"parse_success":False,
                "error":"JEV API unavailable" if API_STATUS!="available" else "MAX_API_CALLS reached",
                "latency_seconds":None,"cache_hit":False,"usage":None}
    request_body={"model":MODEL_ID,"state":make_payload(sample,rep),
                  "questions":{"equipment_state":DECISION_QUESTION}}
    start=time.perf_counter(); CALLS+=1
    try:
        raw=openrouter_json(DECISIONS_URL,request_body,timeout=60)
        content=json.dumps(raw.get("answers"),ensure_ascii=False)
        parsed,ok,err=parse_decision(raw)
        rec={"sample_id":str(sample.sample_id),"representation":rep,"request_timestamp":stamp,
             "raw_response":raw,"raw_content":content,"parsed_response":parsed,"parse_success":ok,
             "error":err,"latency_seconds":time.perf_counter()-start,"cache_hit":False,
             "usage":raw.get("usage"),"model":raw.get("model",MODEL_ID),
             "provider":raw.get("provider"),"requested_model":MODEL_ID,"repeat_tag":repeat_tag}
        if not ok: API_ERRORS+=1
    except Exception as e:
        API_ERRORS+=1
        rec={"sample_id":str(sample.sample_id),"representation":rep,"request_timestamp":stamp,
             "raw_response":None,"raw_content":None,"parsed_response":None,"parse_success":False,
             "error":f"{type(e).__name__}: {str(e).replace(API_KEY or '', '[REDACTED]')}",
             "latency_seconds":time.perf_counter()-start,"cache_hit":False,"usage":None,
             "model":MODEL_ID,"provider":None,"requested_model":MODEL_ID,"repeat_tag":repeat_tag}
    path.write_text(json.dumps(rec,ensure_ascii=False,indent=2,default=str),encoding="utf-8")
    return rec

print("Decision question saved:",OUT/"prompt_v4.txt","| cache:",CACHE)

## 6. One-call JEV smoke test

Send one normal RAW feature payload to pinned Jev 1.13. Confirm that OpenRouter returns the dated Jev 1.13 model and TypeSafe provider. Inspect the native score, probabilities, confidence and reported usage/cost before making additional requests.

In [ ]:
first_target=targets[0]
normal_candidates=samples.loc[(samples.target==first_target)&(samples.region=="normal")]
SINGLE_SMOKE_RESULT=None
if RUN_SINGLE_SMOKE and API_STATUS=="available" and not normal_candidates.empty:
    single_sample=normal_candidates.iloc[0]
    one_result=call_jev(single_sample,"raw")
    SINGLE_SMOKE_RESULT={**one_result,"target":single_sample.target,
                         "region":single_sample.region,"usage_cost":
                         ((one_result.get("usage") or {}).get("cost"))}
    display(pd.DataFrame([{
        "requested_model":MODEL_ID,"returned_model":SINGLE_SMOKE_RESULT.get("model"),
        "provider":SINGLE_SMOKE_RESULT.get("provider"),
        "sample_id":SINGLE_SMOKE_RESULT["sample_id"],"actual_region_local_only":SINGLE_SMOKE_RESULT["region"],
        "raw_content":SINGLE_SMOKE_RESULT.get("raw_content"),"parsed_response":SINGLE_SMOKE_RESULT.get("parsed_response"),
        "parse_success":SINGLE_SMOKE_RESULT.get("parse_success"),"usage":SINGLE_SMOKE_RESULT.get("usage"),
        "latency_seconds":SINGLE_SMOKE_RESULT.get("latency_seconds"),"error":SINGLE_SMOKE_RESULT.get("error")
    }]))
SINGLE_SMOKE_OK=bool(SINGLE_SMOKE_RESULT and SINGLE_SMOKE_RESULT.get("parse_success"))
print("Single JEV smoke parsed:",SINGLE_SMOKE_OK,"| inference calls:",CALLS,"| cache hits:",CACHE_HITS)

## 7. Representation smoke test

After reviewing the single Jev 1.13 response and usage, enable the small smoke set: one normal and one near-failure sample under each representation. It checks the native Decisions response schema and input sensitivity before the full small sample.


In [ ]:
smoke_rows=[]
smoke_samples=[samples.loc[(samples.target==first_target)&(samples.region==r)].iloc[0]
               for r in ("normal","near_failure")
               if not samples.loc[(samples.target==first_target)&(samples.region==r)].empty]
if RUN_REPRESENTATION_SMOKE and SINGLE_SMOKE_OK and API_STATUS=="available" and len(smoke_samples)==2:
    for s in smoke_samples:
        for rep in REPRESENTATIONS:
            smoke_rows.append({**call_jev(s,rep),"target":s.target,"region":s.region,"ttf_seconds":s.ttf_seconds})
else:
    print("Representation smoke gated until the single JEV response is reviewed and valid.")
smoke=pd.DataFrame(smoke_rows)
if not smoke.empty:
    display(smoke[["sample_id","region","representation","raw_content","parsed_response","parse_success","error","latency_seconds","cache_hit"]])
SMOKE_OK=API_STATUS=="available" and len(smoke)==6 and bool(smoke.parse_success.all())
print("Smoke pass:",SMOKE_OK,"| calls:",CALLS,"| cache hits:",CACHE_HITS)

## 8. Run the small experiment

Review both smoke stages, set `RUN_FULL_EXPERIMENT=True` in the setup cell, then run this cell. Smoke rows are already cached. Failed calls are recorded individually; the loop continues.


In [ ]:
experiment=[]
print("Projected unique requests:",len(samples)*len(REPRESENTATIONS),
      "| projected repeats: 6 | MAX_API_CALLS:",MAX_API_CALLS)
if not RUN_FULL_EXPERIMENT:
    print("Full experiment gated. Set RUN_FULL_EXPERIMENT=True after reviewing the smoke output.")
elif not SMOKE_OK:
    print("Full run skipped because the smoke test did not pass.")
elif API_STATUS!="available":
    print("JEV API unavailable; data/sample/scaler/payload prep is available without synthetic responses.")
else:
    for _,s in samples.iterrows():
        for rep in REPRESENTATIONS:
            r=call_jev(s,rep)
            experiment.append({**r,"target":s.target,"target_label":s.target_label,"region":s.region,
                "ttf_seconds":float(s.ttf_seconds),"Tool":s.Tool,"source_file":s.source_file,
                "sequence_index":s.sequence_index,"window_index":int(s.window_index)})
    results=pd.DataFrame(experiment)
    if not results.empty:
        for column,key in (("risk_score","risk_score"),("confidence","confidence"),
                           ("predicted_state","state"),("native_score","native_score")):
            results[column]=results.parsed_response.map(
                lambda value:value.get(key) if isinstance(value,dict) else np.nan)
        for rep in REPRESENTATIONS:
            results.loc[results.representation==rep].to_csv(OUT/f"{rep}_results.csv",index=False)
        results.to_csv(OUT/"all_results.csv",index=False)
        for t in targets:
            display(results.loc[(results.target==t)&(results.representation=="raw"),
                ["sample_id","region","risk_score","confidence","predicted_state"]])
    print("Inference calls:",CALLS,"| cache hits:",CACHE_HITS,"| errors:",API_ERRORS,
          "| parsed:",int(results.parse_success.sum()) if not results.empty else 0)

## 9. Region ordering, ranking metrics and visualization

ROC-AUC/PR-AUC are binary sanity checks (near-failure positive, normal negative); boundary is omitted. No threshold is tuned. These descriptive metrics are not a validation benchmark.

In [ ]:
def score_table(frame):
    if frame.empty: return pd.DataFrame()
    p=frame.loc[frame.parse_success.fillna(False)].copy()
    p["risk"]=p.parsed_response.map(lambda x:x.get("risk_score") if isinstance(x,dict) else np.nan)
    p["confidence"]=p.parsed_response.map(lambda x:x.get("confidence") if isinstance(x,dict) else np.nan)
    rows=[]
    for t in targets:
        for rep in REPRESENTATIONS:
            d=p.loc[(p.target==t)&(p.representation==rep)]
            all_d=frame.loc[(frame.target==t)&(frame.representation==rep)]
            means=d.groupby("region").risk.mean()
            b=d.loc[d.region.isin(["normal","near_failure"])].copy()
            b["positive"]=(b.region=="near_failure").astype(int)
            auc=float(roc_auc_score(b.positive,b.risk)) if b.positive.nunique()==2 else np.nan
            ap=float(average_precision_score(b.positive,b.risk)) if b.positive.nunique()==2 else np.nan
            rho=float(spearmanr(d.risk,np.log10(d.ttf_seconds.clip(lower=1))).statistic) if len(d)>=3 else np.nan
            rv=d.risk.to_numpy(dtype=float)
            rows.append({"target":t,"representation":rep,"normal_mean":means.get("normal",np.nan),
                "boundary_mean":means.get("boundary",np.nan),"near_failure_mean":means.get("near_failure",np.nan),
                "separation":means.get("near_failure",np.nan)-means.get("normal",np.nan),
                "ordered":bool(means.get("normal",np.nan)<means.get("boundary",np.nan)<means.get("near_failure",np.nan))
                    if all(r in means.index for r in REGIONS) else False,
                "ROC_AUC_near_vs_normal":auc,"PR_AUC_near_vs_normal":ap,"Spearman_risk_log10_TTF":rho,
                "risk_std":float(np.std(rv,ddof=1)) if len(rv)>1 else np.nan,
                "confidence_mean":float(d.confidence.mean()) if len(d) else np.nan,
                "confidence_std":float(d.confidence.std(ddof=1)) if len(d)>1 else np.nan,
                "near_half_fraction":float(np.mean(np.abs(rv-.5)<=.05)) if len(rv) else np.nan,
                "extreme_fraction":float(np.mean((rv<=.05)|(rv>=.95)) ) if len(rv) else np.nan,
                "parse_rate":float(all_d.parse_success.fillna(False).mean()) if len(all_d) else np.nan,"n_parsed":len(d)})
    return pd.DataFrame(rows)

if "results" in globals() and not results.empty:
    summary=score_table(results)
    summary.to_csv(OUT/"representation_summary.csv",index=False)
    display(summary)
    p=results.loc[results.parse_success.fillna(False)].copy()
    p["risk"]=p.parsed_response.map(lambda x:x.get("risk_score") if isinstance(x,dict) else np.nan)
    p["confidence"]=p.parsed_response.map(lambda x:x.get("confidence") if isinstance(x,dict) else np.nan)
    region_risk_summary=p.groupby(["target","representation","region"],as_index=False).agg(
        risk_mean=("risk","mean"),risk_median=("risk","median"),risk_std=("risk","std"),
        risk_min=("risk","min"),risk_max=("risk","max"),n=("risk","size"),
        confidence_mean=("confidence","mean"),confidence_std=("confidence","std"))
    region_risk_summary.to_csv(OUT/"region_risk_summary.csv",index=False)
    display(region_risk_summary)
    display(p.pivot_table(index=["target_label","sample_id","region"],columns="representation",values="risk").reset_index().head(18))
    fig,axes=plt.subplots(1,3,figsize=(15,4),sharey=True)
    for ax,t in zip(axes,targets):
        d=p[p.target==t]
        for i,rep in enumerate(REPRESENTATIONS):
            z=d[d.representation==rep]
            xx=z.region.map({"normal":0,"boundary":1,"near_failure":2}).to_numpy(float)+(i-1)*.08
            ax.scatter(xx,z.risk,s=16,alpha=.55,label=rep)
        ax.set_xticks([0,1,2],["Normal","Boundary","Near failure"]); ax.set_title(LABELS[t]); ax.set_ylim(-.03,1.03); ax.grid(axis="y",alpha=.2)
    axes[0].set_ylabel("JEV risk score"); axes[-1].legend(frameon=False); fig.tight_layout()
    fig.savefig(OUT/"risk_by_region.png",dpi=150,bbox_inches="tight"); display(fig); plt.close(fig)
    fig,axes=plt.subplots(1,3,figsize=(15,4),sharey=True)
    for ax,t in zip(axes,targets):
        d=p[(p.target==t)&(p.ttf_seconds>0)]
        for rep in REPRESENTATIONS:
            z=d[d.representation==rep]; ax.scatter(np.log10(z.ttf_seconds),z.risk,s=16,alpha=.55,label=rep)
        ax.set_title(LABELS[t]); ax.set_xlabel("log10(TTF seconds)"); ax.grid(alpha=.2)
    axes[0].set_ylabel("JEV risk score"); axes[-1].legend(frameon=False); fig.tight_layout()
    fig.savefig(OUT/"risk_vs_log_ttf.png",dpi=150,bbox_inches="tight"); display(fig); plt.close(fig)
else:
    summary=pd.DataFrame()
    print("Full experiment has not been run.")

## 10. Repeatability, parser review and 05b recommendation

Repeat two RAW samples per region for the first target once. Confidence is reported as model output, not treated as calibrated probability.

In [ ]:
stability=[]
if RUN_FULL_EXPERIMENT and "results" in globals() and not results.empty:
    for r in REGIONS:
        for _,s in samples.loc[(samples.target==first_target)&(samples.region==r)].head(2).iterrows():
            again=call_jev(s,"raw",repeat_tag="stability_repeat_1")
            base=results.loc[(results.sample_id==s.sample_id)&(results.representation=="raw")]
            a=base.iloc[0].parsed_response.get("risk_score") if not base.empty and isinstance(base.iloc[0].parsed_response,dict) else np.nan
            b=again.get("parsed_response",{}).get("risk_score") if isinstance(again.get("parsed_response"),dict) else np.nan
            pair=np.asarray([a,b],dtype=float); pair=pair[np.isfinite(pair)]
            stability.append({"target":first_target,"sample_id":s.sample_id,"region":r,
                "baseline_risk":a,"repeat_risk":b,"mean":float(pair.mean()) if len(pair) else np.nan,
                "std":float(pair.std(ddof=1)) if len(pair)>1 else np.nan,
                "range":float(np.ptp(pair)) if len(pair)>1 else np.nan,
                "parse_success":again.get("parse_success"),"cache_hit":again.get("cache_hit")})
stability=pd.DataFrame(stability)
if not stability.empty:
    stability.to_csv(OUT/"stability_results.csv",index=False); display(stability)
    print("Mean paired repeat SD:",stability["std"].mean())

if "results" in globals() and not results.empty:
    failures=results.loc[~results.parse_success.fillna(False),["target","sample_id","representation","error","raw_content"]]
    display(results.groupby("representation",as_index=False).agg(
        requests=("sample_id","size"),parsed=("parse_success","sum"),cache_hits=("cache_hit","sum"),
        median_latency_seconds=("latency_seconds","median"),errors=("error",lambda x:int(x.notna().sum()))))
    display(failures.head(20))
    if not summary.empty:
        average=summary.groupby("representation",as_index=False).agg(
            separation=("separation","mean"),ordering_fraction=("ordered","mean"),
            ROC_AUC=("ROC_AUC_near_vs_normal","mean"),PR_AUC=("PR_AUC_near_vs_normal","mean"),
            risk_std=("risk_std","mean"),near_half_fraction=("near_half_fraction","mean"),
            extreme_fraction=("extreme_fraction","mean"),parse_rate=("parse_rate","mean"),
            confidence_mean=("confidence_mean","mean"),confidence_std=("confidence_std","mean"))
        display(average)
        usable=average[average.parse_rate>0].copy()
        if len(usable):
            usable["rank_score"]=usable.ROC_AUC.fillna(.5)+usable.separation.fillna(0)-.05*usable.risk_std.fillna(1)
            recommendation=str(usable.sort_values("rank_score",ascending=False).iloc[0].representation).upper()
        else: recommendation="No parsed representation"
    else: recommendation="Pending"
else:
    failures=pd.DataFrame(); average=pd.DataFrame(); recommendation="Pending smoke review/full run"
print("Recommended input representation for larger JEV experiment:",recommendation)

## 11. Save artifacts

Saved: selected_samples.csv, train_scaler_stats.json, per-representation result CSVs, all_results.csv, representation_summary.csv, region_risk_summary.csv, stability_results.csv, prompt/config, plots and raw response cache. No credential is written.

In [ ]:
if "results" in globals() and not results.empty:
    results.to_csv(OUT/"all_results.csv",index=False)
    for rep in REPRESENTATIONS: results[results.representation==rep].to_csv(OUT/f"{rep}_results.csv",index=False)
if "summary" in globals() and not summary.empty: summary.to_csv(OUT/"representation_summary.csv",index=False)
if "region_risk_summary" in globals() and not region_risk_summary.empty: region_risk_summary.to_csv(OUT/"region_risk_summary.csv",index=False)
if not stability.empty: stability.to_csv(OUT/"stability_results.csv",index=False)
cache_records=[json.loads(p.read_text(encoding="utf-8")) for p in CACHE.glob("*.json")]
reported_costs=[float((r.get("usage") or {}).get("cost")) for r in cache_records
                if (r.get("usage") or {}).get("cost") is not None]
Returned_models=pd.Series([r.get("model",MODEL_ID) for r in cache_records]).value_counts().to_dict()
Providers=pd.Series([r.get("provider") for r in cache_records]).value_counts().to_dict()
REPORTED_COST_USD=float(sum(reported_costs)) if reported_costs else None
config={
 "experiment":"05a JEV input calibration tutorial","prompt_version":PROMPT_VERSION,
 "model_id":MODEL_ID,"model_name":"TypeSafe: Jev 1.13",
 "model_response_verified":SINGLE_SMOKE_OK,"openrouter_decisions_url":DECISIONS_URL,
 "openrouter_decisions_docs_url":DECISIONS_DOCS_URL,
 "decision_type":"score","risk_mapping":"Jev native score / 2","state_mapping":"maximum probability level",
 "credential_env_name":"JEV_API_KEY or OPENROUTER_API_KEY","credential_saved":False,
 "compact_feature_count":len(features),"feature_columns":features,"target_columns":targets,
 "fixed_split_artifact":str(split_path.relative_to(ROOT)),"split_group_definition":meta["group_definition"],
 "split_seed":SEED,"sample_split":"validation groups only",
 "ttf_sampling_bands_seconds":{"near_failure":"<=5000","boundary":"5000<TTF<=50000","normal":">50000"},
 "requested_samples_per_target_region":N_PER_REGION,"sample_counts":sample_counts.to_dict(),
 "representations":list(REPRESENTATIONS),"quantiles_approximate":True,"quantile_stride":QUANTILE_STRIDE,
 "max_api_calls":MAX_API_CALLS,"inference_calls_this_execution":CALLS,
 "unique_cached_inference_responses":len(cache_records),"openrouter_reported_cost_usd":REPORTED_COST_USD,
 "openrouter_returned_model_counts":Returned_models,"openrouter_provider_counts":Providers,
 "cache_hits_this_execution":CACHE_HITS,"api_errors_or_parse_failures_this_execution":API_ERRORS,
 "full_experiment_enabled":RUN_FULL_EXPERIMENT,"recommended_input_for_larger_experiment":recommendation}
(OUT/"config.json").write_text(json.dumps(config,indent=2,ensure_ascii=False,default=str),encoding="utf-8")
print("Artifacts:",OUT)
print("Inference calls this execution:",CALLS,"| cache hits:",CACHE_HITS,"| errors:",API_ERRORS)
print("Unique Jev 1.13 responses:",len(cache_records),"| OpenRouter-reported total cost USD:",REPORTED_COST_USD)
print("Returned model counts:",Returned_models,"| providers:",Providers)

## 12. Observed result and next experiment

Snapshot from this small run: 45 validation samples, five in each TTF region for each of three fault targets. The table gives the mean of the three target-wise region means and the mean target-wise near-failure versus normal ROC-AUC / PR-AUC. These are descriptive sanity checks, not a benchmark.

| Input | Normal risk | Boundary risk | Near-failure risk | ROC-AUC | PR-AUC | Targets in order |
|---|---:|---:|---:|---:|---:|---:|
| RAW | 0.374 | 0.413 | 0.386 | 0.540 | 0.685 | 0/3 |
| Z-score | 0.490 | 0.484 | 0.599 | 0.667 | 0.776 | 2/3 |
| Robust | 0.408 | 0.472 | 0.642 | 0.913 | 0.945 | 3/3 |

**Recommended input representation for larger JEV experiment: Robust.** The RAW response was compressed and did not order the three TTF regions for any target. Z-score improved two targets but reversed the pressure-high ordering. Robust ordered all three targets, with a mean near-minus-normal gap of 0.234. No result is treated as ground truth, and no threshold was tuned.

The native score confidence remained low (mean 0.033 RAW, 0.255 Z-score, 0.189 Robust). All 135 sample/representation responses parsed. In six repeated RAW pairs for the first target, mean paired standard deviation was 0.0153 and the largest absolute change was 0.030. No risk was at the 0/1 extremes. The largest absolute Robust-scaled input was 21.84, so inspect those feature outliers before extending the sample.

The following section tests whether a single pressure feature can improve Jev separation when its already train-only Robust-scaled value is multiplied globally by 0.5× or 2×. Labels are used only for offline evaluation; each Jev request contains the same 20 numeric features and the same decision question. Treat any apparent improvement as exploratory until it is reproduced on broader grouped validation. The execution flags above are reset to `False` for staged reruns; displayed outputs and cached responses are from this completed run.



## 13. Jev 1.13 single-feature scaling sensitivity

Compare one globally scaled pressure feature at 0.5×, 1×, and 2× across the same 45 fixed validation samples. The 1× arm is the fresh Jev control; labels and region names stay local for evaluation.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, math, os, time, urllib.error, urllib.request
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import average_precision_score, roc_auc_score

def find_root(start=None):
    for p in (Path(start or Path.cwd()).resolve(), *Path(start or Path.cwd()).resolve().parents):
        if (p / "artifacts/jev_tut/config.json").is_file(): return p
    raise FileNotFoundError("05a Jev 1.13 artifacts were not found")

ROOT=find_root()
BASE=ROOT/"artifacts/jev_tut"
OUT=ROOT/"artifacts/jev_scale_sensitivity"
CACHE=OUT/"cache"
OUT.mkdir(parents=True,exist_ok=True)
CACHE.mkdir(parents=True,exist_ok=True)
MODEL_ID="typesafe/jev-1.13"
DECISIONS_URL="https://openrouter.ai/api/alpha/decisions"
PROMPT_VERSION="jev_scale_v1"
FEATURE_TO_SCALE="FLOWCOOLPRESSURE__shape_factor"
FACTORS={"down_0p5":0.5,"control_1p0":1.0,"up_2p0":2.0}
VARIANTS=tuple(FACTORS)
REGIONS=("normal","boundary","near_failure")
MAX_API_CALLS=150
RUN_SINGLE_SMOKE=True
RUN_REPRESENTATION_SMOKE=False
RUN_FULL_EXPERIMENT=False

def load_env_keys(path):
    if path.is_file():
        for line in path.read_text(encoding="utf-8").splitlines():
            s=line.strip()
            if s and not s.startswith("#") and "=" in s:
                k,v=s.split("=",1)
                k,v=k.strip(),v.strip().strip('"').strip("'")
                if k in {"JEV_API_KEY","OPENROUTER_API_KEY"} and v and not os.environ.get(k):
                    os.environ[k]=v

load_env_keys(ROOT/".env")
API_KEY=os.environ.get("JEV_API_KEY") or os.environ.get("OPENROUTER_API_KEY")
print("Project:",ROOT)
print("Jev 1.13 credential available:",bool(API_KEY))
print("Planned distinct calls: 45 samples × 3 factors + 6 control repeats = 141; cap:",MAX_API_CALLS)

### 13.1 Reuse and audit 05a artifacts

Re-use 03's 20 selected compact features, 05a's train-only scaling statistics, fixed grouped validation samples, native Decisions question and saved Robust baseline. No feature selection or full dataset scan is repeated here.

In [ ]:
base_config=json.loads((BASE/"config.json").read_text(encoding="utf-8"))
if base_config["model_id"]!=MODEL_ID or base_config["compact_feature_count"]!=20:
    raise ValueError("05a model or feature count does not match this study")
features=list(base_config["feature_columns"])
targets=list(base_config["target_columns"])
if FEATURE_TO_SCALE not in features: raise ValueError("Perturbed feature is not in 03 compact set")
samples=pd.read_csv(BASE/"selected_samples.csv",low_memory=False)
old_robust=pd.read_csv(BASE/"robust_results.csv",low_memory=False)
stats_doc=json.loads((BASE/"train_scaler_stats.json").read_text(encoding="utf-8"))
scaler=stats_doc["targets"]
if stats_doc["source"]!="03 train groups only": raise ValueError("Scaler is not train-only")
if len(samples)!=45 or samples.sample_id.duplicated().any(): raise ValueError("Unexpected 05a sample set")
if set(samples.sample_id)!=set(old_robust.sample_id) or not old_robust.parse_success.all():
    raise ValueError("05a Robust baseline is incomplete or uses different samples")
counts=samples.groupby(["target","region"]).size().unstack(fill_value=0).reindex(columns=REGIONS,fill_value=0)
if not (counts==5).all().all(): raise ValueError("Expected five samples per target and region")
split=pd.read_csv(ROOT/"artifacts/random_forest/ttf_baseline_group_split.csv",usecols=["group_id","split"])
valid_groups=set(split.loc[split.split=="validation","group_id"].astype(str))
sample_groups=samples.source_file.astype(str)+"::"+samples.sequence_index.astype(str)
if not set(sample_groups).issubset(valid_groups): raise ValueError("Sample outside fixed validation groups")
selected=pd.read_csv(ROOT/"artifacts/random_forest/jev_candidate_features.csv")
display(selected.loc[selected.feature==FEATURE_TO_SCALE,["feature","importance_reason"]])
display(counts)
print("Samples:",len(samples),"| feature count:",len(features),"| distinct groups:",sample_groups.nunique())

### 13.2 Fixed input change and native Jev response

The question text is loaded from 05a's saved prompt file. A request contains only `model`, `state.features`, and the fixed `questions` object. `risk_score` is Jev's native 0–2 score divided by two; `confidence` and level probabilities are used as returned.

In [ ]:
prompt_text=(BASE/"prompt_v4.txt").read_text(encoding="utf-8")
QUESTION=json.loads(prompt_text.split("question=\n",1)[1])
QUESTION_HASH=hashlib.sha256(json.dumps(QUESTION,sort_keys=True).encode()).hexdigest()
assert QUESTION["type"]=="score" and len(QUESTION["criteria"])==3
(OUT/"decision_question.json").write_text(json.dumps(QUESTION,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")

def robust_values(sample):
    values={}
    for feature in features:
        spec=scaler[str(sample.target)][feature]
        value=(float(sample[feature])-spec["median_approx"])/spec["safe_iqr"]
        if not math.isfinite(value): raise ValueError(f"Non-finite Robust input: {feature}")
        values[feature]=value
    return values

def make_request(sample,variant):
    values=robust_values(sample)
    values[FEATURE_TO_SCALE]*=FACTORS[variant]
    assert len(values)==20 and set(values)==set(features)
    return {"model":MODEL_ID,"state":{"features":values},
            "questions":{"equipment_state":QUESTION}}

def parse_decision(raw):
    answer=(raw.get("answers") or {}).get("equipment_state") or {}
    try:
        if not str(raw.get("model","")).startswith(MODEL_ID): raise ValueError("returned model is not Jev 1.13")
        if raw.get("provider")!="TypeSafe": raise ValueError("returned provider is not TypeSafe")
        if answer.get("type")!="score": raise ValueError("unexpected decision type")
        score=float(answer["score"]); confidence=float(answer["confidence"])
        probabilities={str(i):float(answer["probabilities"][str(i)]) for i in range(3)}
        if not math.isfinite(score) or not 0<=score<=2: raise ValueError("score outside [0,2]")
        if not math.isfinite(confidence) or not 0<=confidence<=1: raise ValueError("confidence outside [0,1]")
        if any(not math.isfinite(v) or not 0<=v<=1 for v in probabilities.values()):
            raise ValueError("invalid level probabilities")
        if not .95<=sum(probabilities.values())<=1.05: raise ValueError("probabilities do not sum to one")
    except (KeyError,TypeError,ValueError) as exc: return answer,False,str(exc)
    state=("normal","uncertain","degrading")[max(range(3),key=lambda i:probabilities[str(i)])]
    return {"risk_score":score/2,"confidence":confidence,"state":state,
            "native_score":score,"probabilities":probabilities},True,None

def cache_file(sample,variant,repeat_tag=""):
    identity="|".join([str(sample.target),str(sample.sample_id),variant,
                       PROMPT_VERSION,MODEL_ID,QUESTION_HASH,repeat_tag])
    return CACHE/(hashlib.sha256(identity.encode()).hexdigest()+".json")

CALLS=0; CACHE_HITS=0; API_ERRORS=0
def call_jev(sample,variant,repeat_tag=""):
    global CALLS,CACHE_HITS,API_ERRORS
    path=cache_file(sample,variant,repeat_tag)
    if path.is_file():
        saved=json.loads(path.read_text(encoding="utf-8")); saved["cache_hit"]=True; CACHE_HITS+=1
        return saved
    stamp=datetime.now(timezone.utc).isoformat()
    rec={"sample_id":str(sample.sample_id),"representation":variant,
         "request_timestamp":stamp,"requested_model":MODEL_ID,"repeat_tag":repeat_tag,
         "feature":FEATURE_TO_SCALE,"factor":FACTORS[variant],"cache_hit":False}
    if not API_KEY or CALLS>=MAX_API_CALLS:
        return {**rec,"parse_success":False,"error":"Jev API unavailable or MAX_API_CALLS reached",
                "raw_response":None,"parsed_response":None,"usage":None,"latency_seconds":None}
    body=make_request(sample,variant)
    req=urllib.request.Request(DECISIONS_URL,
        data=json.dumps(body,ensure_ascii=False,allow_nan=False).encode("utf-8"),
        headers={"Authorization":f"Bearer {API_KEY}","Content-Type":"application/json"},method="POST")
    start=time.perf_counter(); CALLS+=1
    try:
        with urllib.request.urlopen(req,timeout=60) as response:
            raw=json.loads(response.read().decode("utf-8"))
        parsed,ok,error=parse_decision(raw)
        rec.update({"raw_response":raw,"parsed_response":parsed,"parse_success":ok,"error":error,
                    "usage":raw.get("usage"),"model":raw.get("model"),"provider":raw.get("provider")})
        if not ok: API_ERRORS+=1
    except urllib.error.HTTPError as exc:
        API_ERRORS+=1
        response_text=exc.read().decode("utf-8",errors="replace")
        rec.update({"raw_response":response_text,"parsed_response":None,"parse_success":False,
                    "error":f"HTTP {exc.code}","usage":None,"model":None,"provider":None})
    except Exception as exc:
        API_ERRORS+=1
        rec.update({"raw_response":None,"parsed_response":None,"parse_success":False,
                    "error":f"{type(exc).__name__}: {str(exc).replace(API_KEY,'[REDACTED]')}",
                    "usage":None,"model":None,"provider":None})
    rec["latency_seconds"]=time.perf_counter()-start
    path.write_text(json.dumps(rec,ensure_ascii=False,indent=2,default=str),encoding="utf-8")
    return rec

test_values=[robust_values(s)[FEATURE_TO_SCALE] for _,s in samples.iterrows()]
print("Affected Robust feature range:",float(min(test_values)),"to",float(max(test_values)))
print("After 2×, maximum absolute value:",float(max(abs(2*v) for v in test_values)))
print("Question SHA256:",QUESTION_HASH,"| response cache:",CACHE)

### 13.3 Staged API smoke

First call one normal control sample. Inspect the returned model/provider, score, probabilities, confidence and cost. Then use one normal and one near-failure sample under all three factors before running the full set.

In [ ]:
first_target=targets[0]
normal=samples.loc[(samples.target==first_target)&(samples.region=="normal")].iloc[0]
single=call_jev(normal,"control_1p0") if RUN_SINGLE_SMOKE else None
SINGLE_SMOKE_OK=bool(single and single.get("parse_success"))
if single:
    display(pd.DataFrame([{"sample_id":single["sample_id"],"requested_model":MODEL_ID,
        "returned_model":single.get("model"),"provider":single.get("provider"),
        "parsed":single.get("parsed_response"),"usage":single.get("usage"),
        "latency_seconds":single.get("latency_seconds"),"error":single.get("error")}]))
print("Single Jev 1.13 smoke parsed:",SINGLE_SMOKE_OK,"| calls:",CALLS,"| cache hits:",CACHE_HITS)

In [ ]:
RUN_REPRESENTATION_SMOKE=bool(SINGLE_SMOKE_OK)  # Enabled only after reviewing the single-call output.
print("Three-factor smoke enabled:",RUN_REPRESENTATION_SMOKE)

In [ ]:
smoke_rows=[]
smoke_samples=[samples.loc[(samples.target==first_target)&(samples.region==r)].iloc[0]
               for r in ("normal","near_failure")]
if RUN_REPRESENTATION_SMOKE and SINGLE_SMOKE_OK:
    for sample in smoke_samples:
        for variant in VARIANTS:
            rec=call_jev(sample,variant)
            parsed=rec.get("parsed_response") or {}
            smoke_rows.append({"sample_id":sample.sample_id,"region":sample.region,
                "variant":variant,"risk_score":parsed.get("risk_score"),
                "confidence":parsed.get("confidence"),"state":parsed.get("state"),
                "model":rec.get("model"),"provider":rec.get("provider"),
                "parse_success":rec.get("parse_success"),"error":rec.get("error"),
                "cost":(rec.get("usage") or {}).get("cost"),"cache_hit":rec.get("cache_hit")})
smoke=pd.DataFrame(smoke_rows)
if not smoke.empty: display(smoke)
SMOKE_OK=len(smoke)==6 and bool(smoke.parse_success.all())
print("Three-factor smoke pass:",SMOKE_OK,"| calls:",CALLS,"| cache hits:",CACHE_HITS)

In [ ]:
RUN_FULL_EXPERIMENT=bool(SMOKE_OK)  # Enabled only after reviewing the six-call output.
print("Full small experiment enabled:",RUN_FULL_EXPERIMENT)

### 13.4 Paired 45-sample comparison

Call the same samples under all three factors. The API sees only the 20 compact feature values. Target, TTF and region are joined to responses locally for analysis. Smoke results use the same cache keys.

In [ ]:
rows=[]
if RUN_FULL_EXPERIMENT and SMOKE_OK:
    for _,sample in samples.iterrows():
        for variant in VARIANTS:
            rec=call_jev(sample,variant)
            parsed=rec.get("parsed_response") or {}
            rows.append({**rec,"target":sample.target,"target_label":sample.target_label,
                "region":sample.region,"ttf_seconds":sample.ttf_seconds,
                "Tool":sample.Tool,"source_file":sample.source_file,
                "sequence_index":sample.sequence_index,"window_index":sample.window_index,
                "risk_score":parsed.get("risk_score"),"confidence":parsed.get("confidence"),
                "predicted_state":parsed.get("state"),"native_score":parsed.get("native_score")})
else:
    print("Full experiment remains gated after smoke review.")
results=pd.DataFrame(rows)
if not results.empty:
    results.to_csv(OUT/"all_results.csv",index=False)
    print("Comparison rows:",len(results),"| parsed:",int(results.parse_success.sum()),
          "| calls:",CALLS,"| cache hits:",CACHE_HITS,"| errors:",API_ERRORS)
    display(results.groupby(["target","region","representation"]).size().unstack(fill_value=0))

### 13.5 Region separation, ranking and paired response changes

ROC-AUC and PR-AUC compare near-failure to normal; boundary stays in ordering and separation checks. They are descriptive for this tiny fixed sample. Paired risk changes show whether a factor changes fault-like samples differently from normal ones.

In [ ]:
summary=pd.DataFrame(); macro=pd.DataFrame(); region_stats=pd.DataFrame()
paired_delta=pd.DataFrame(); historical_drift=pd.DataFrame()
recommendation="Pending"
if not results.empty:
    parsed=results.loc[results.parse_success.fillna(False)].copy()
    region_stats=parsed.groupby(["target","representation","region"],as_index=False).agg(
        risk_mean=("risk_score","mean"),risk_median=("risk_score","median"),
        risk_std=("risk_score","std"),risk_min=("risk_score","min"),
        risk_max=("risk_score","max"),confidence_mean=("confidence","mean"),
        n=("risk_score","size"))
    report=[]
    for target in targets:
        for variant in VARIANTS:
            all_d=results.loc[(results.target==target)&(results.representation==variant)]
            d=parsed.loc[(parsed.target==target)&(parsed.representation==variant)]
            means=d.groupby("region").risk_score.mean()
            binary=d.loc[d.region.isin(("normal","near_failure"))].copy()
            binary["is_near"]=(binary.region=="near_failure").astype(int)
            auc=float(roc_auc_score(binary.is_near,binary.risk_score)) if binary.is_near.nunique()==2 else np.nan
            ap=float(average_precision_score(binary.is_near,binary.risk_score)) if binary.is_near.nunique()==2 else np.nan
            report.append({"target":target,"representation":variant,
                "normal_mean":means.get("normal",np.nan),"boundary_mean":means.get("boundary",np.nan),
                "near_failure_mean":means.get("near_failure",np.nan),
                "separation":means.get("near_failure",np.nan)-means.get("normal",np.nan),
                "ordered":bool(means.get("normal",np.nan)<means.get("boundary",np.nan)<means.get("near_failure",np.nan))
                    if all(r in means.index for r in REGIONS) else False,
                "ROC_AUC":auc,"PR_AUC":ap,"parse_rate":float(all_d.parse_success.fillna(False).mean()),
                "confidence_mean":float(d.confidence.mean()) if len(d) else np.nan,"n_parsed":len(d)})
    summary=pd.DataFrame(report)
    macro=summary.groupby("representation",as_index=False).agg(
        normal_mean=("normal_mean","mean"),boundary_mean=("boundary_mean","mean"),
        near_failure_mean=("near_failure_mean","mean"),separation=("separation","mean"),
        ordered_targets=("ordered","sum"),ROC_AUC=("ROC_AUC","mean"),PR_AUC=("PR_AUC","mean"),
        parse_rate=("parse_rate","mean"),confidence_mean=("confidence_mean","mean"))
    pivot=parsed.pivot_table(index=["target","sample_id","region"],columns="representation",values="risk_score").reset_index()
    delta_rows=[]
    for variant in ("down_0p5","up_2p0"):
        for _,row in pivot.iterrows():
            if pd.notna(row.get(variant)) and pd.notna(row.get("control_1p0")):
                delta_rows.append({"target":row.target,"sample_id":row.sample_id,"region":row.region,
                                   "representation":variant,"delta_risk":row[variant]-row["control_1p0"]})
    paired_delta=pd.DataFrame(delta_rows).groupby(["target","representation","region"],as_index=False).agg(
        mean_delta=("delta_risk","mean"),std_delta=("delta_risk","std"),n=("delta_risk","size"))
    historical=old_robust[["sample_id","risk_score"]].rename(columns={"risk_score":"previous_05a_risk"})
    historical_drift=pivot[["target","sample_id","region","control_1p0"]].merge(historical,on="sample_id")
    historical_drift["control_minus_05a"]=historical_drift.control_1p0-historical_drift.previous_05a_risk
    base=macro.set_index("representation").loc["control_1p0"]
    options=macro.loc[(macro.representation!="control_1p0") &
        (macro.separation>base.separation) & (macro.ROC_AUC>=base.ROC_AUC) &
        (macro.ordered_targets>=base.ordered_targets)]
    recommendation=(str(options.sort_values("separation",ascending=False).iloc[0].representation)
                    if len(options) else "control_1p0")
    display(summary)
    display(macro)
    display(paired_delta)
    print("Mean absolute fresh-control minus 05a Robust risk:",historical_drift.control_minus_05a.abs().mean())
    print("Exploratory candidate under separation/AUC/ordering rule:",recommendation)
    region_stats.to_csv(OUT/"region_stats.csv",index=False)
    summary.to_csv(OUT/"target_comparison.csv",index=False)
    macro.to_csv(OUT/"macro_comparison.csv",index=False)
    paired_delta.to_csv(OUT/"paired_delta_by_region.csv",index=False)
    historical_drift.to_csv(OUT/"fresh_control_vs_05a.csv",index=False)

### 13.6 Repeatability, plots, and saved record

Repeat two 1.0-control rows per region for the first fault target. Keep full raw Jev responses, parsed decisions, prompt version, scaling factor, usage/cost and errors in the cache. No threshold is chosen.

In [ ]:
repeat_rows=[]
if RUN_FULL_EXPERIMENT and not results.empty:
    for region in REGIONS:
        for _,sample in samples.loc[(samples.target==first_target)&(samples.region==region)].head(2).iterrows():
            rec=call_jev(sample,"control_1p0",repeat_tag="repeat_1")
            original=results.loc[(results.sample_id==sample.sample_id)&
                                 (results.representation=="control_1p0")].iloc[0]
            repeat_risk=(rec.get("parsed_response") or {}).get("risk_score")
            values=np.asarray([original.risk_score,repeat_risk],dtype=float)
            values=values[np.isfinite(values)]
            repeat_rows.append({"target":first_target,"sample_id":sample.sample_id,"region":region,
                "control_risk":original.risk_score,"repeat_risk":repeat_risk,
                "paired_std":float(values.std(ddof=1)) if len(values)==2 else np.nan,
                "paired_range":float(np.ptp(values)) if len(values)==2 else np.nan,
                "parse_success":rec.get("parse_success"),"cache_hit":rec.get("cache_hit")})
repeatability=pd.DataFrame(repeat_rows)
if not repeatability.empty:
    repeatability.to_csv(OUT/"repeatability.csv",index=False)
    display(repeatability)
    print("Mean paired control SD:",repeatability.paired_std.mean())

if not results.empty:
    fig,axes=plt.subplots(1,3,figsize=(15,4),sharey=True)
    positions={"normal":0,"boundary":1,"near_failure":2}
    for ax,target in zip(axes,targets):
        for i,variant in enumerate(VARIANTS):
            d=results.loc[(results.target==target)&(results.representation==variant)&results.parse_success]
            x=d.region.map(positions).to_numpy(float)+(i-1)*.08
            ax.scatter(x,d.risk_score,s=17,alpha=.65,label=variant)
        ax.set_xticks([0,1,2],["Normal","Boundary","Near failure"])
        ax.set_title(target.removeprefix("ttf_").removesuffix("_seconds"))
        ax.set_ylim(-.03,1.03); ax.grid(axis="y",alpha=.2)
    axes[0].set_ylabel("Jev 1.13 risk score"); axes[-1].legend(frameon=False); fig.tight_layout()
    fig.savefig(OUT/"risk_by_region_and_factor.png",dpi=150,bbox_inches="tight"); display(fig); plt.close(fig)
    if not paired_delta.empty:
        d=paired_delta.groupby(["representation","region"],as_index=False).mean_delta.mean()
        fig,ax=plt.subplots(figsize=(7,4))
        for variant in ("down_0p5","up_2p0"):
            v=d[d.representation==variant].set_index("region").reindex(REGIONS)
            ax.plot(REGIONS,v.mean_delta,marker="o",label=variant)
        ax.axhline(0,color="black",linewidth=.8); ax.set_ylabel("Mean paired risk change vs 1× control")
        ax.grid(alpha=.2); ax.legend(frameon=False); fig.tight_layout()
        fig.savefig(OUT/"paired_risk_change.png",dpi=150,bbox_inches="tight"); display(fig); plt.close(fig)

cached=[json.loads(p.read_text(encoding="utf-8")) for p in CACHE.glob("*.json")]
reported_cost=sum(float((r.get("usage") or {}).get("cost") or 0) for r in cached)
config={"experiment":"05b single-feature Jev 1.13 scale sensitivity",
    "model_id":MODEL_ID,"endpoint":DECISIONS_URL,"prompt_version":PROMPT_VERSION,
    "question_sha256":QUESTION_HASH,"feature":FEATURE_TO_SCALE,"factors":FACTORS,
    "sample_count":len(samples),"sample_source":"05a fixed validation sample",
    "scaler_source":"05a train-only Robust stats","feature_count":len(features),
    "max_api_calls":MAX_API_CALLS,"network_calls_this_execution":CALLS,
    "cache_hits_this_execution":CACHE_HITS,"unique_cached_responses":len(cached),
    "api_or_parse_errors_this_execution":API_ERRORS,"reported_cost_usd":reported_cost,
    "returned_models":pd.Series([r.get("model") for r in cached]).value_counts().to_dict(),
    "providers":pd.Series([r.get("provider") for r in cached]).value_counts().to_dict(),
    "full_experiment_enabled":RUN_FULL_EXPERIMENT,
    "exploratory_candidate":recommendation,"credential_saved":False}
(OUT/"config.json").write_text(json.dumps(config,ensure_ascii=False,indent=2,default=str),encoding="utf-8")
print("Saved to:",OUT)
print("Jev 1.13 calls this execution:",CALLS,"| cache hits:",CACHE_HITS,
      "| unique raw responses:",len(cached),"| errors:",API_ERRORS)
print("OpenRouter-reported cost USD:",reported_cost)

### 13.7 Observed comparison and next validation

The 0.5× and 2× factors are fixed for all samples and affect only one feature. Compare target-level ordering, near-minus-normal risk, ROC-AUC / PR-AUC, paired regional risk changes, confidence, and fresh-control drift from 05a. A larger grouped sample or untouched holdout is required before retaining a factor. Jev decisions remain support signals, not measured fault labels.

**Observed result (45 samples; five per target and TTF region):**

| Feature factor | Mean near-failure minus normal risk | Mean ROC-AUC | Mean PR-AUC | Targets ordered |
|---|---:|---:|---:|---:|
| 0.5x | 0.242 | 0.927 | 0.943 | 2/3 |
| 1x control | **0.249** | **0.973** | **0.981** | **3/3** |
| 2x | 0.230 | 0.933 | 0.956 | 2/3 |

Neither scaling factor improved the overall comparison. Keep the feature at 1x for now. The 0.5x factor slightly increased separation for pressure-high, but reversed its three-region ordering; 2x reduced separation across all three targets. This small run does not support tuning the feature scale.

All 135 paired comparison responses parsed. The six fresh control repeats had a mean paired standard deviation of 0.0124 risk points; the mean absolute difference between fresh 1x controls and the original 05a Robust scores was 0.0219. Treat small shifts as response variability until reproduced on broader grouped validation. Across all 141 cached responses, the returned model was `typesafe/jev-1.13-20260917` from `TypeSafe`; reported total cost was $0.00583.


## 14. Extreme pressure-shape scaling: 0.1× and 10×

This extension uses the same 45 fixed group-validation windows, 20 train-only Robust-transformed features, Jev 1.13 model, score question, and single scaled feature (`FLOWCOOLPRESSURE__shape_factor`) as section 13. Only the selected feature receives a 0.1× or 10× multiplier. A fresh 1× control is called on the same windows to measure response drift. A six-response pilot checks model, provider, parsing and usage before the remaining calls. The extension uses its own success cache and USD 0.01/135-attempt caps; prior 05a and section-13 artifacts are retained. The default notebook view below reads only the saved results and makes no API requests.


In [ ]:
from pathlib import Path
import sys
EXT_ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                 if (p / "artifacts/jev_tut/config.json").is_file()), None)
if EXT_ROOT is None:
    raise FileNotFoundError("Project root with saved 05a artifacts is required.")
ext_site = EXT_ROOT / ".venv" / "Lib" / "site-packages"
if ext_site.is_dir():
    sys.path.insert(0, str(ext_site))
sys.path.insert(0, str(EXT_ROOT / "src"))
import pandas as pd
from IPython.display import display, Image
from fabjudge.jev_scale_extremes import run_scale_extension
ext_result = run_scale_extension(EXT_ROOT, allow_network=False)
EXT_OUT = EXT_ROOT / "artifacts" / "jev_scale_sensitivity" / "extreme_0p1_10x"
print("New responses:", ext_result["new_request_count"], "| parse errors:", ext_result["parse_errors"],
      "| input/output tokens:", ext_result["input_tokens_new_requests"], ext_result["output_tokens_new_requests"],
      "| reported cost USD:", ext_result["reported_cost_usd_new_requests"])
print("Returned models:", ext_result["models"], "| providers:", ext_result["providers"])
display(pd.read_csv(EXT_OUT / "macro_comparison.csv"))
display(pd.read_csv(EXT_OUT / "target_comparison.csv"))
display(pd.read_csv(EXT_OUT / "paired_group_bootstrap_ci.csv"))
display(pd.read_csv(EXT_OUT / "usage_summary.csv"))
display(Image(filename=str(EXT_OUT / "risk_by_region_0p1_to_10x.png")))
display(Image(filename=str(EXT_OUT / "paired_extreme_scale_change.png")))


### Interpretation

For the newly called arms, the macro near-failure-minus-normal risk gap is 0.249 at 0.1×, 0.255 at the fresh 1× control, and 0.216 at 10×. ROC-AUC is 0.913, 0.933, and 0.880, respectively; PR-AUC is 0.945, 0.956, and 0.925. The three-region ordering holds for 3/3, 3/3, and 1/3 targets. The original 0.5×/1×/2× arms were recorded in an earlier batch and appear separately in the comparison table.

The paired group-bootstrap separation changes versus fresh 1× are −0.006 for 0.1× (95% interval −0.024 to 0.011) and −0.039 for 10× (−0.067 to −0.011). Fresh 1× scores differ from the historical 1× responses by a mean absolute 0.0207, so cross-batch comparisons also contain response variation. The 10× arm sends scaled feature values as large as 50.3 in absolute value. With five windows per target/TTF band, these are small-sample sensitivity results. Retain the 1× input as the exploratory baseline; Jev outputs are decision support and no threshold is tuned here.
